In [7]:
from collections import Counter

In [12]:
text = """We present an approach for using machine learning to automatically discover the governing equations and hidden
properties of real physical systems from observations. We train a “graph neural network” to simulate the dynamics of
our solar system’s Sun, planets, and large moons from 30 years of trajectory data. We then use symbolic regression
to discover an analytical expression for the force law implicitly learned by the neural network, which our results
showed is equivalent to Newton’s law of gravitation. The key assumptions that were required were translational and
rotational equivariance, and Newton’s second and third laws of motion. Our approach correctly discovered the form
of the symbolic force law. Furthermore, our approach did not require any assumptions about the masses of planets
and moons or physical constants. They, too, were accurately inferred through our methods. Though, of course, the
classical law of gravitation has been known since Isaac Newton, our result serves as a validation that our method can
discover unknown laws and hidden properties from observed data. More broadly this work represents a key step
toward realizing the potential of machine learning for accelerating scientific discovery."""

In [14]:
"A".encode(), "脑".encode() 

(b'A', b'\xe8\x84\x91')

In [29]:
words = Counter(tuple(w.encode()) for w in text.split())
words

Counter({(111, 102): 10,
         (116, 104, 101): 9,
         (97, 110, 100): 7,
         (111, 117, 114): 6,
         (116, 111): 4,
         (87, 101): 3,
         (97, 112, 112, 114, 111, 97, 99, 104): 3,
         (102, 111, 114): 3,
         (100, 105, 115, 99, 111, 118, 101, 114): 3,
         (102, 114, 111, 109): 3,
         (97,): 3,
         (108, 97, 119): 3,
         (119, 101, 114, 101): 3,
         (97, 110): 2,
         (109, 97, 99, 104, 105, 110, 101): 2,
         (108, 101, 97, 114, 110, 105, 110, 103): 2,
         (104, 105, 100, 100, 101, 110): 2,
         (112, 114, 111, 112, 101, 114, 116, 105, 101, 115): 2,
         (112, 104, 121, 115, 105, 99, 97, 108): 2,
         (110, 101, 117, 114, 97, 108): 2,
         (109, 111, 111, 110, 115): 2,
         (100, 97, 116, 97, 46): 2,
         (115, 121, 109, 98, 111, 108, 105, 99): 2,
         (102, 111, 114, 99, 101): 2,
         (78, 101, 119, 116, 111, 110, 226, 128, 153, 115): 2,
         (107, 101, 121): 2,
         (9

In [16]:
bytes([111, 102]).decode(), bytes([116,104,101]).decode()

('of', 'the')

In [30]:
# token ID: atomic bytes
# here are the more atomic ones
vocab = {i: bytes([i]) for i in range(256)}
merges = {}

In [21]:
def merge(w, pair, new_id):
    out, i = [], 0
    while i < len(w):
        if i < len(w) - 1 and (w[i], w[i+1]) == pair:
            out.append(new_id); i += 2
        else:
            out.append(w[i]); i += 1
    return tuple(out)

merge((116, 104, 101), (104, 101), 257)

(116, 257)

In [31]:
# sets vocab size which already has 256
# trade offs here...embeddings need to grow
num_merges = 50

# train
for new_id in range(256, 256 + num_merges):
    pairs = Counter()

    # all words found so far
    for w, freq in words.items():
        # pairing up neighboring chars in a word
        # (116, 104, 101) -> (116,104),(104,101)
        for p in zip(w, w[1:]):
            pairs[p] += freq
    if not pairs:
        break

    # most common (x, x) pair in all words
    best = max(pairs, key=pairs.get)


    # most common substr
    merges[best] = new_id
    vocab[new_id] = vocab[best[0]] + vocab[best[1]]

    # new redesign the text
    words = Counter({merge(w, best, new_id): f for w, f in words.items()})
    
len(vocab)

306

In [ ]:
def encode(word):
    byte_enc = tuple(word.encode())
    while True:
        # if theres a merge use it
        pairs = [p for p in zip(byte_enc, byte_enc[1:]) if p in merges]
        if not pairs:
            return list(byte_enc)

		# keep going until theres no more
        best = min(pairs, key=merges.get) # lower index ^^frequency
        byte_enc = merge(byte_enc, best, merges[best])


encode("there")

[268, 256]

In [33]:
def decode(ids):
    return b"".join(vocab[i] for i in ids).decode()
    
decode([268, 256])

'there'

___

In [4]:
import requests
from bs4 import BeautifulSoup

link = "https://www.gutenberg.org/cache/epub/100/pg100-images.html"
html = requests.get(link, timeout=10).text
soup = BeautifulSoup(html, "html.parser")
for tag in soup(["script", "style"]):
	tag.decompose()
wordbabble = "\n".join(line.strip() for line in soup.get_text("\n").splitlines() if line.strip())

In [5]:
with open("shakespeare.txt", "a", encoding="utf-8") as f:
    f.write(wordbabble + "\n")

___

CONDENSED

In [35]:
import tiktoken
import json
import numpy as np

enc = tiktoken.encoding_for_model("gpt-4o")

# read and dump in chunks
with open("shakespeare.txt", encoding="utf-8") as f, open("input.bin", "wb") as out:
    for line in f:
        np.array(enc.encode(line), dtype=np.uint32).tofile(out)

enc.n_vocab

200019

In [44]:
import json

with open("shakespeare.txt", encoding="utf-8") as f, open("tiktoken.json", "w") as out:
	ids = [t for line in f for t in enc.encode(line)]
	json.dump(ids, out)

OR

In [6]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel()
tok.train(["shakespeare.txt"], trainers.BpeTrainer(vocab_size=32000, special_tokens=["<pad>", "<eos>"]))
tok.save("tokenizer.json")